# 迁移学习

实现三种迁移学习策略，将源域建筑训练的LSTM模型迁移到目标域建筑。

## 迁移策略
1. **预训练-微调（Full Fine-tuning）**：在源域数据上预训练模型，然后在目标域上微调所有参数
2. **冻结部分层微调（Partial Fine-tuning）**：冻结模型前几层（LSTM特征提取层），仅微调靠近输出的层
3. **冻结特征提取（Frozen Feature Extractor）**：将预训练模型作为固定特征提取器，训练新的回归层

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt
import joblib

# 从 src 模块导入共享配置和函数
from src import (
    # preprocessing
    load_data,
    create_sequences,
    # models
    LSTMPredictor,
    FeatureExtractorRegressor,
    # training
    set_seed,
    evaluate,
    LoadDataset,
    EarlyStopping,
    run_epoch,
    predict,
    # visualization
    setup_plot_style,
    plot_training_history,
    plot_predictions,
    # config
    BASE_DIR,
    MODEL_DIR,
    SCALER_DIR,
    FIGURES_DIR,
    TARGET_COL,
    TIME_COL,
    SEED,
)

# 超参数配置
LOOKBACK = 24   # 输入窗口长度（小时）
HORIZON = 1     # 预测步长（小时）
BATCH_SIZE = 72
EPOCHS = 100
HIDDEN_SIZE = 128

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

setup_plot_style()
plt.rcParams['font.family'] = 'DejaVu Sans'

set_seed(SEED)

## 数据准备

In [2]:
# 数据路径
SOURCE_TRAIN_PATH = BASE_DIR / 'source_train_std.csv'  # 源域全量数据
TARGET_TRAIN_PATH = BASE_DIR / 'target_train_std.csv'
TARGET_VAL_PATH = BASE_DIR / 'target_val_std.csv'
TARGET_TEST_PATH = BASE_DIR / 'target_test_std.csv'

# 读取源域数据（全量数据）
source_full_df = load_data(SOURCE_TRAIN_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)

# 从源域全量数据中划分部分作为验证集（用于早停）
source_val_ratio = 0.2
source_n = len(source_full_df)
source_val_end = int(source_n * source_val_ratio)

source_train_df = source_full_df.iloc[:-source_val_end].reset_index(drop=True)
source_val_df = source_full_df.iloc[-source_val_end:].reset_index(drop=True)

# 读取目标域数据
target_train_df = load_data(TARGET_TRAIN_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)
target_val_df = load_data(TARGET_VAL_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)
target_test_df = load_data(TARGET_TEST_PATH).dropna(subset=[TARGET_COL]).reset_index(drop=True)

# 确定特征列（除时间戳外）
FEATURE_COLS = [c for c in source_train_df.columns if c != TIME_COL]
input_size = len(FEATURE_COLS)

# 构造序列
X_source_train, y_source_train = create_sequences(source_train_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON)
X_source_val, y_source_val = create_sequences(source_val_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON)

X_target_train, y_target_train = create_sequences(target_train_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON)
X_target_val, y_target_val = create_sequences(target_val_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON)
X_target_test, y_target_test = create_sequences(target_test_df, FEATURE_COLS, TARGET_COL, LOOKBACK, HORIZON)

print(f"　源域训练序列: {X_source_train.shape}")
print(f"　源域验证序列: {X_source_val.shape}")
print(f"\n目标域训练序列: {X_target_train.shape}")
print(f"目标域验证序列: {X_target_val.shape}")
print(f"目标域测试序列: {X_target_test.shape}")

　源域训练序列: (6920, 24, 45)
　源域验证序列: (1732, 24, 45)

目标域训练序列: (1497, 24, 45)
目标域验证序列: (307, 24, 45)
目标域测试序列: (308, 24, 45)


In [3]:
# 创建数据加载器
source_train_dataset = LoadDataset(X_source_train, y_source_train)
source_val_dataset = LoadDataset(X_source_val, y_source_val)
source_train_loader = DataLoader(source_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
source_val_loader = DataLoader(source_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

target_train_dataset = LoadDataset(X_target_train, y_target_train)
target_val_dataset = LoadDataset(X_target_val, y_target_val)
target_test_dataset = LoadDataset(X_target_test, y_target_test)
target_train_loader = DataLoader(target_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
target_val_loader = DataLoader(target_val_dataset, batch_size=BATCH_SIZE, shuffle=False)
target_test_loader = DataLoader(target_test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 加载目标域的scaler
target_scaler = joblib.load(SCALER_DIR / 'target_load_scaler.joblib')

# 定义损失函数
criterion = nn.L1Loss()  # MAE loss

## 在源域上训练基础模型

In [ ]:
# 初始化模型
base_model = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE, horizon=HORIZON).to(device)

# 优化器和调度器
optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-3, weight_decay=1e-4)
lr_scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=4, min_lr=1e-5)
early_stopping = EarlyStopping(patience=5, restore_best_weights=True)

train_losses = []
val_losses = []
best_epoch = 0
for epoch in range(EPOCHS):
    train_loss = run_epoch(base_model, source_train_loader, criterion, device, optimizer)
    val_loss = run_epoch(base_model, source_val_loader, criterion, device)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    lr_scheduler.step(val_loss)
    
    if early_stopping(val_loss, base_model, epoch):
        break

early_stopping.restore(base_model)

# 输出最佳训练信息
best_info = early_stopping.get_best_info()
print(f"源域预训练完成，共 {len(train_losses)} 个 epoch")
print(f"最佳 epoch: {best_info['best_epoch']}, 最佳 val_loss: {best_info['best_loss']:.6f}")

# 保存预训练模型
torch.save(base_model.state_dict(), MODEL_DIR / 'pretrained_source.pt')
print(f"预训练模型已保存: {MODEL_DIR / 'pretrained_source.pt'}")

# 可视化训练过程（保存到figures目录）
plot_training_history(train_losses, val_losses, 'Source Domain Training History', 
                      save_path=FIGURES_DIR / '源域预训练_损失曲线.png')

## 迁移学习实验

### Baseline

作为对照组，不使用任何预训练权重，从零开始在目标域数据上训练模型。

In [ ]:
baseline_model = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE, horizon=HORIZON).to(device)

optimizer_baseline = torch.optim.Adam(baseline_model.parameters(), lr=1e-3, weight_decay=1e-4)
early_stopping_baseline = EarlyStopping(patience=5, restore_best_weights=True)
lr_scheduler_baseline = ReduceLROnPlateau(optimizer_baseline, factor=0.5, patience=4, min_lr=1e-5)

train_losses_baseline = []
val_losses_baseline = []

for epoch in range(EPOCHS):
    train_loss = run_epoch(baseline_model, target_train_loader, criterion, device, optimizer_baseline)
    val_loss = run_epoch(baseline_model, target_val_loader, criterion, device)
    
    train_losses_baseline.append(train_loss)
    val_losses_baseline.append(val_loss)
    
    lr_scheduler_baseline.step(val_loss)
    
    if early_stopping_baseline(val_loss, baseline_model, epoch):
        break

early_stopping_baseline.restore(baseline_model)

# 输出最佳训练信息
best_info_baseline = early_stopping_baseline.get_best_info()
print(f"Baseline训练完成，共 {len(train_losses_baseline)} 个 epoch")
print(f"最佳 epoch: {best_info_baseline['best_epoch']}, 最佳 val_loss: {best_info_baseline['best_loss']:.6f}")

torch.save(baseline_model.state_dict(), MODEL_DIR / 'transfer_baseline.pt')

# 在目标域测试集上评估
y_pred_baseline = predict(baseline_model, target_test_loader, device)
metrics_baseline = evaluate(y_target_test, y_pred_baseline, target_scaler)

print("\n基线模型测试集性能：")
print(f"MAE: {metrics_baseline['MAE']:.4f}")
print(f"RMSE: {metrics_baseline['RMSE']:.4f}")
print(f"CV-RMSE: {metrics_baseline['CV-RMSE']:.4f}%")
print(f"MAPE: {metrics_baseline['MAPE']:.4f}%")
print(f"R2: {metrics_baseline['R2']:.4f}")

plot_training_history(train_losses_baseline, val_losses_baseline, 'Baseline Model Training History',
                      save_path=FIGURES_DIR / 'Baseline_损失曲线.png')
plot_predictions(y_target_test, y_pred_baseline, 'Baseline Model Predictions', target_scaler,
                 save_path=FIGURES_DIR / 'Baseline_预测对比.png')

### 策略1：预训练-微调（Full Fine-tuning）

加载源域预训练模型，在目标域训练集上微调所有参数。

In [ ]:
# 加载预训练模型
finetuned_full = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE, horizon=HORIZON).to(device)
finetuned_full.load_state_dict(torch.load(MODEL_DIR / 'pretrained_source.pt', map_location=device))

# 微调所有参数
optimizer_ft_full = torch.optim.Adam(finetuned_full.parameters(), lr=1e-4, weight_decay=1e-4)
early_stopping_ft_full = EarlyStopping(patience=5, restore_best_weights=True)
lr_scheduler_ft_full = ReduceLROnPlateau(optimizer_ft_full, factor=0.5, patience=4, min_lr=1e-6)

train_losses_ft_full = []
val_losses_ft_full = []

for epoch in range(EPOCHS):
    train_loss = run_epoch(finetuned_full, target_train_loader, criterion, device, optimizer_ft_full)
    val_loss = run_epoch(finetuned_full, target_val_loader, criterion, device)
    
    train_losses_ft_full.append(train_loss)
    val_losses_ft_full.append(val_loss)
    
    lr_scheduler_ft_full.step(val_loss)
    
    if early_stopping_ft_full(val_loss, finetuned_full, epoch):
        break

early_stopping_ft_full.restore(finetuned_full)

# 输出最佳训练信息
best_info_ft_full = early_stopping_ft_full.get_best_info()
print(f"全参数微调完成，共 {len(train_losses_ft_full)} 个 epoch")
print(f"最佳 epoch: {best_info_ft_full['best_epoch']}, 最佳 val_loss: {best_info_ft_full['best_loss']:.6f}")

torch.save(finetuned_full.state_dict(), MODEL_DIR / 'transfer_full_finetune.pt')

# 在目标域测试集上评估
y_pred_ft_full = predict(finetuned_full, target_test_loader, device)
metrics_ft_full = evaluate(y_target_test, y_pred_ft_full, target_scaler)

print("\n全参数微调模型测试集性能：")
print(f"MAE: {metrics_ft_full['MAE']:.4f}")
print(f"RMSE: {metrics_ft_full['RMSE']:.4f}")
print(f"CV-RMSE: {metrics_ft_full['CV-RMSE']:.4f}%")
print(f"MAPE: {metrics_ft_full['MAPE']:.4f}%")
print(f"R2: {metrics_ft_full['R2']:.4f}")

plot_training_history(train_losses_ft_full, val_losses_ft_full, 'Full Fine-tuning Training History',
                      save_path=FIGURES_DIR / '全参数微调_损失曲线.png')
plot_predictions(y_target_test, y_pred_ft_full, 'Full Fine-tuning Predictions', target_scaler,
                 save_path=FIGURES_DIR / '全参数微调_预测对比.png')

### 策略2：冻结部分层微调（Partial Fine-tuning）

冻结LSTM层（特征提取层），仅微调全连接层。

In [ ]:
# 加载预训练模型
finetuned_partial = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE, horizon=HORIZON).to(device)
finetuned_partial.load_state_dict(torch.load(MODEL_DIR / 'pretrained_source.pt', map_location=device))

# 冻结LSTM层，只微调fc层
for name, param in finetuned_partial.named_parameters():
    if 'lstm' in name:
        param.requires_grad = False
        print(f"  冻结: {name}")
    else:
        print(f"  可训练: {name}")

# 微调（只微调fc层）
optimizer_ft_partial = torch.optim.Adam(
    filter(lambda p: p.requires_grad, finetuned_partial.parameters()), 
    lr=1e-3, weight_decay=1e-4
)
early_stopping_ft_partial = EarlyStopping(patience=5, restore_best_weights=True)
lr_scheduler_ft_partial = ReduceLROnPlateau(optimizer_ft_partial, factor=0.5, patience=4, min_lr=1e-6)

train_losses_ft_partial = []
val_losses_ft_partial = []

for epoch in range(EPOCHS):
    train_loss = run_epoch(finetuned_partial, target_train_loader, criterion, device, optimizer_ft_partial)
    val_loss = run_epoch(finetuned_partial, target_val_loader, criterion, device)
    
    train_losses_ft_partial.append(train_loss)
    val_losses_ft_partial.append(val_loss)
    
    lr_scheduler_ft_partial.step(val_loss)
    
    if early_stopping_ft_partial(val_loss, finetuned_partial, epoch):
        break

early_stopping_ft_partial.restore(finetuned_partial)

# 输出最佳训练信息
best_info_ft_partial = early_stopping_ft_partial.get_best_info()
print(f"冻结部分层微调完成，共 {len(train_losses_ft_partial)} 个 epoch")
print(f"最佳 epoch: {best_info_ft_partial['best_epoch']}, 最佳 val_loss: {best_info_ft_partial['best_loss']:.6f}")

torch.save(finetuned_partial.state_dict(), MODEL_DIR / 'transfer_partial_finetune.pt')

# 在目标域测试集上评估
y_pred_ft_partial = predict(finetuned_partial, target_test_loader, device)
metrics_ft_partial = evaluate(y_target_test, y_pred_ft_partial, target_scaler)

print("\n冻结部分层微调模型测试集性能：")
print(f"MAE: {metrics_ft_partial['MAE']:.4f}")
print(f"RMSE: {metrics_ft_partial['RMSE']:.4f}")
print(f"CV-RMSE: {metrics_ft_partial['CV-RMSE']:.4f}%")
print(f"MAPE: {metrics_ft_partial['MAPE']:.4f}%")
print(f"R2: {metrics_ft_partial['R2']:.4f}")

plot_training_history(train_losses_ft_partial, val_losses_ft_partial, 'Partial Fine-tuning Training History',
                      save_path=FIGURES_DIR / '冻结LSTM微调_损失曲线.png')
plot_predictions(y_target_test, y_pred_ft_partial, 'Partial Fine-tuning Predictions', target_scaler,
                 save_path=FIGURES_DIR / '冻结LSTM微调_预测对比.png')

### 策略3：冻结特征提取（Frozen Feature Extractor）

将预训练模型作为固定特征提取器，训练新的回归层。

In [ ]:
# 加载预训练模型
pretrained = LSTMPredictor(input_size=input_size, hidden_size=HIDDEN_SIZE, horizon=HORIZON).to(device)
pretrained.load_state_dict(torch.load(MODEL_DIR / 'pretrained_source.pt', map_location=device))

# 创建特征提取器模型
feature_extractor_model = FeatureExtractorRegressor(pretrained, horizon=HORIZON).to(device)

# 只训练新添加的层
optimizer_fe = torch.optim.Adam(
    filter(lambda p: p.requires_grad, feature_extractor_model.parameters()), 
    lr=1e-3, weight_decay=1e-4
)
early_stopping_fe = EarlyStopping(patience=5, restore_best_weights=True)
lr_scheduler_fe = ReduceLROnPlateau(optimizer_fe, factor=0.5, patience=4, min_lr=1e-6)

train_losses_fe = []
val_losses_fe = []

for epoch in range(EPOCHS):
    train_loss = run_epoch(feature_extractor_model, target_train_loader, criterion, device, optimizer_fe)
    val_loss = run_epoch(feature_extractor_model, target_val_loader, criterion, device)
    
    train_losses_fe.append(train_loss)
    val_losses_fe.append(val_loss)
    
    lr_scheduler_fe.step(val_loss)
    
    if early_stopping_fe(val_loss, feature_extractor_model, epoch):
        break

early_stopping_fe.restore(feature_extractor_model)

# 输出最佳训练信息
best_info_fe = early_stopping_fe.get_best_info()
print(f"冻结特征提取训练完成，共 {len(train_losses_fe)} 个 epoch")
print(f"最佳 epoch: {best_info_fe['best_epoch']}, 最佳 val_loss: {best_info_fe['best_loss']:.6f}")

torch.save(feature_extractor_model.state_dict(), MODEL_DIR / 'transfer_feature_extractor.pt')

# 在目标域测试集上评估
y_pred_fe = predict(feature_extractor_model, target_test_loader, device)
metrics_fe = evaluate(y_target_test, y_pred_fe, target_scaler)

print("\n冻结特征提取模型测试集性能：")
print(f"MAE: {metrics_fe['MAE']:.4f}")
print(f"RMSE: {metrics_fe['RMSE']:.4f}")
print(f"CV-RMSE: {metrics_fe['CV-RMSE']:.4f}%")
print(f"MAPE: {metrics_fe['MAPE']:.4f}%")
print(f"R2: {metrics_fe['R2']:.4f}")

plot_training_history(train_losses_fe, val_losses_fe, 'Frozen Feature Extractor Training History',
                      save_path=FIGURES_DIR / '冻结特征提取_损失曲线.png')
plot_predictions(y_target_test, y_pred_fe, 'Frozen Feature Extractor Predictions', target_scaler,
                 save_path=FIGURES_DIR / '冻结特征提取_预测对比.png')

## 实验结果汇总

In [9]:
# 汇总所有方法的性能
summary_df = pd.DataFrame([
    {'策略': 'Baseline', **metrics_baseline},
    {'策略': '全参数微调', **metrics_ft_full},
    {'策略': '冻结LSTM微调', **metrics_ft_partial},
    {'策略': '冻结特征提取', **metrics_fe},
])

print(summary_df.to_string(index=False))

# 保存结果
summary_df.to_csv(BASE_DIR / 'transfer_learning_results.csv', index=False, encoding='utf-8-sig')
print(f"\n结果已保存到 {BASE_DIR / 'transfer_learning_results.csv'}")

      策略      MAE      RMSE     MAPE       R2
Baseline 9.109162 12.946689 7.201392 0.898193
   全参数微调 7.159620 11.017163 5.785107 0.926277
冻结LSTM微调 7.807880 12.381965 6.356403 0.906881
  冻结特征提取 7.242007 11.318797 5.771329 0.922185

结果已保存到 data\transfer_learning_results.csv


## 结果分析

根据实验结果分析各迁移学习策略的效果：

1. **Baseline (从零训练)**：作为对照组，展示在目标域数据有限情况下的模型性能

2. **Full Fine-tuning (全参数微调)**：利用源域预训练权重作为初始化，在目标域上微调所有参数。适合目标域数据量适中的情况。

3. **Partial Fine-tuning (冻结LSTM)**：冻结特征提取层，仅微调输出层。适合目标域数据量较小的情况。

4. **Frozen Feature Extractor (冻结特征提取)**：将预训练模型作为固定特征提取器，仅训练新的回归层。适合目标域数据量极小的情况。